# Đọc chart doanh thu Online Retail II: tháng đỉnh, thị trường, và tháng thiếu dữ liệu

Bộ `online-retail-ii`, sheet **`Year 2010-2011`** (541 910 dòng, 01/12/2010 tới 09/12/2011): cửa hàng quà
tặng online ở Anh, khách lẻ lẫn khách sỉ. Sheet này phủ đúng 13 tháng lịch, từ tháng 12/2010 tới tháng
12/2011, và **dừng ở ngày 09/12/2011**.

Bạn nhận ba chart từ đồng nghiệp để duyệt trước buổi họp với CEO: doanh thu theo tháng, tỉ trọng doanh
thu theo quốc gia, và (chart bạn tự thêm) số ngày có giao dịch mỗi tháng. Việc của bạn: viết ba hàm sinh
dữ liệu cho chart, vẽ, rồi **đọc chart theo quy trình 5 bước**, đặc biệt bước 1 (kỳ nào, đủ chưa) và
bước 3 (điểm bất thường là thật hay do dữ liệu). Cuối cùng làm phần trắc nghiệm trong app.

Quy tắc làm sạch (như bài Basic 03), áp dụng trên từng **dòng**: bỏ hoá đơn huỷ (mã bắt đầu bằng `C`,
đổi cột `Invoice` sang `str` trước), bỏ dòng `Quantity <= 0` hoặc `Price <= 0`. Doanh thu một dòng là
`Quantity * Price` (GBP).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/08-visualization-chart-reading/exercises/intermediate/02-retail-chart-reading -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc sheet 2010-2011 (xlsx: khoảng 20-60 giây)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

RETAIL = ensure("online-retail-ii")
tx = pd.read_excel(RETAIL / "online_retail_II.xlsx", sheet_name="Year 2010-2011")
print(tx.shape, tx.InvoiceDate.min(), "→", tx.InvoiceDate.max())
tx.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Doanh thu theo tháng

`monthly_revenue(tx) -> pd.Series`: tổng doanh thu đã làm sạch theo tháng của `InvoiceDate`. Index là
`DatetimeIndex` ngày đầu tháng (`2011-01-01`, `2011-02-01`, ...), liên tục từ tháng đầu tới tháng cuối có
dữ liệu: tháng không có dòng hợp lệ nào vẫn có mặt với giá trị `0`. Tên Series: `revenue`.

Quy tắc làm sạch (như bài Basic 03), áp dụng trên từng **dòng**: bỏ hoá đơn huỷ (mã bắt đầu bằng `C`,
đổi cột `Invoice` sang `str` trước), bỏ dòng `Quantity <= 0` hoặc `Price <= 0`. Doanh thu một dòng là
`Quantity * Price` (GBP).

Nên viết một hàm phụ `_clean(tx)` trả các dòng hợp lệ kèm cột doanh thu; cả ba hàm dùng lại nó.

In [ ]:
def _clean(tx: pd.DataFrame) -> pd.DataFrame:
    """Các dòng hợp lệ, thêm cột revenue = Quantity * Price (không sửa tx)."""
    invoice = tx["Invoice"].astype(str)
    ok = ~invoice.str.startswith("C") & (tx["Quantity"] > 0) & (tx["Price"] > 0)
    return tx.loc[ok].assign(revenue=lambda d: d["Quantity"] * d["Price"])


def monthly_revenue(tx: pd.DataFrame) -> pd.Series:
    """Doanh thu theo tháng (index ngày đầu tháng, liên tục, tháng trống = 0), tên "revenue"."""
    sales = _clean(tx)
    return sales.set_index("InvoiceDate")["revenue"].resample("MS").sum().rename("revenue")

In [ ]:
# explore: chart đồng nghiệp gửi - doanh thu theo tháng, vẽ ngây thơ
rev = monthly_revenue(tx)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(rev.index, rev / 1e6, marker="o", color="#4c78a8")
ax.set_ylim(0, None)
ax.set_xlabel("tháng")
ax.set_ylabel("doanh thu (triệu GBP)")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Doanh thu theo tháng", loc="left")
plt.show()
print((rev / 1e3).round(0).astype(int).to_string())

## 2. Tỉ trọng doanh thu theo quốc gia

`country_share(tx, top) -> pd.Series`: tỉ trọng doanh thu đã làm sạch của từng `Country`, giữ `top` nước
lớn nhất (sắp giảm dần), gộp mọi nước còn lại vào một dòng `"Other"` đặt **cuối cùng**. Không còn nước nào
ngoài top thì không có dòng `"Other"`. Tổng bằng 1. Tên Series: `share`.

In [ ]:
def country_share(tx: pd.DataFrame, top: int) -> pd.Series:
    """Tỉ trọng doanh thu top quốc gia (giảm dần) + "Other" ở cuối, tổng = 1, tên "share"."""
    by_country = _clean(tx).groupby("Country")["revenue"].sum().sort_values(ascending=False)
    share = by_country / by_country.sum()
    head, rest = share.iloc[:top], share.iloc[top:]
    if len(rest):
        head = pd.concat([head, pd.Series({"Other": rest.sum()})])
    return head.rename("share")

In [ ]:
# explore: bar ngang top 5 + Other (không dùng pie: UK lấn át, các lát còn lại không so được)
share = country_share(tx, top=5)
fig, ax = plt.subplots(figsize=(7, 3))
colors = ["#4c78a8"] * (len(share) - 1) + ["#c9ced6"]
bars = ax.barh(share.index, share.values * 100, color=colors)
ax.invert_yaxis()
ax.bar_label(bars, labels=[f"{v:.1%}" for v in share.values], padding=3, fontsize=9)
ax.set_xlim(0, 100)
ax.set_xlabel("% doanh thu 12/2010-12/2011")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Tỉ trọng doanh thu theo quốc gia", loc="left")
plt.show()
share.round(4)

## 3. Số ngày có giao dịch mỗi tháng

`month_coverage(tx) -> pd.Series`: số **ngày lịch khác nhau** có ít nhất một dòng hợp lệ trong mỗi tháng.
Cùng index với `monthly_revenue` (tháng trống = `0`). Tên Series: `days`.

Đây là chart "kiểm chất lượng dữ liệu" ít ai vẽ mà nên vẽ: trước khi tin một cú tăng hay giảm, xem tháng
đó có bao nhiêu ngày dữ liệu. Nó trả lời bước 1 của quy trình đọc chart: kỳ này đã đủ chưa?

In [ ]:
def month_coverage(tx: pd.DataFrame) -> pd.Series:
    """Số ngày có giao dịch hợp lệ mỗi tháng (index như monthly_revenue), tên "days"."""
    days = _clean(tx)["InvoiceDate"].dt.normalize().drop_duplicates()
    return days.groupby(days.dt.to_period("M").dt.to_timestamp()).size().asfreq("MS", fill_value=0).rename("days")

In [ ]:
# explore: số ngày có giao dịch và doanh thu mỗi ngày giao dịch
days = month_coverage(tx)
per_day = (rev / days.replace(0, np.nan)).rename("revenue_per_day")

fig, (a, b) = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
a.bar(days.index, days.values, width=20, color="#9aa5b1")
a.bar_label(a.containers[0], padding=2, fontsize=8)
a.set_ylabel("số ngày có giao dịch")
a.set_ylim(0, 31)
b.plot(per_day.index, per_day / 1e3, marker="o", color="#4c78a8")
b.set_ylim(0, None)
b.set_ylabel("nghìn GBP / ngày giao dịch")
b.set_xlabel("tháng")
for ax in (a, b):
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()
print(pd.DataFrame({"revenue": rev.round(0), "days": days, "per_day": per_day.round(0)}).to_string())

In [ ]:
# explore: vì sao một tháng đủ chỉ có khoảng 26 ngày giao dịch? Đếm ngày có giao dịch theo thứ trong tuần
trading_days = _clean(tx)["InvoiceDate"].dt.normalize().drop_duplicates()
names = ["T2", "T3", "T4", "T5", "T6", "T7", "CN"]
print(trading_days.dt.day_of_week.value_counts().reindex(range(7), fill_value=0).set_axis(names).to_string())

## 4. Đọc ba chart, rồi làm trắc nghiệm

Đọc theo quy trình 5 bước và **ghi riêng quan sát với kết luận**:

1. **Trục và kỳ**: tháng đầu và tháng cuối có đủ ngày không? Một tháng "đủ" ở cửa hàng này có bao nhiêu
   ngày giao dịch, và vì sao không phải 30?
2. **Hình dạng**: doanh thu đi ngang hay có mùa vụ? Đỉnh ở đâu?
3. **Điểm bất thường**: điểm cuối của chart doanh thu là thật hay do dữ liệu? Ngoài tháng cuối, còn tháng
   nào ít ngày giao dịch hơn hẳn, và lý do khả dĩ (nghỉ lễ)?
4. **So với tham chiếu**: so tháng 12/2011 với tháng 11/2011 bằng một đại lượng công bằng (doanh thu mỗi
   ngày giao dịch). Lưu ý doanh thu mỗi ngày cũng có mùa vụ trong tháng: đầu tháng 12 là cao điểm đặt
   hàng Giáng sinh, nên đây vẫn là so sánh có giới hạn.
5. **Câu hỏi tiếp theo**: UK chiếm phần lớn doanh thu; tăng trưởng mùa cuối năm đến từ UK hay nước ngoài?

Viết lại tiêu đề chart doanh thu theo tháng thành headline insight cho CEO, và quyết định cách vẽ điểm
tháng 12/2011.

Xong thì mở app, làm phần **trắc nghiệm** của bài.